# معمل تحديثات LangChain الحديثة وإعداد بيئة UV

أهلاً بك في معمل المحاضرة 57!

في هذا المعمل الخفيف، هنستكشف التحديثات الكبيرة اللي عملتها LangChain في الإصدار الحديث (`v1+`) وهنجرب أدوات التطوير الجديدة:
1. التأكد من أداة **`uv`** السريعة والمبنية بلغة Rust.
2. فحص مكتبات LangChain الحديثة المثبتة عندك ومقارنة إصداراتها.
3. استكشاف أنواع الرسائل الأربعة الأساسية في الوكلاء (`System`, `Human`, `AI`, `Tool`).
4. نظرة على وحدة الـ **Middleware** الجديدة.


In [1]:
# فحص بيئة بايثون وأداة uv السريعة
import sys
import subprocess

uv_check = subprocess.run(["uv", "--version"], capture_output=True, text=True)
print("إصدار أداة uv لديك:", uv_check.stdout.strip())
print("إصدار بايثون النشط:", sys.version.split()[0])


إصدار أداة uv لديك: uv 0.13.1 (62faed335 2026-10-13 x86_64-pc-windows-msvc)
إصدار بايثون النشط: 3.12.12


### 1. فحص إصدار LangChain واستكشاف الأدوات الحديثة

لاحظ إننا شغالين دلوقتي على الجيل الأول المحدث (`LangChain v1+`) اللي دمج كل الأدوات بشكل منظم مع LangGraph. تعال نشوف الإصدارات عندك.


In [2]:
import langchain
import langchain_core
import pydantic

print(f"LangChain version: {langchain.__version__}")
print(f"LangChain Core version: {langchain_core.__version__}")
print(f"Pydantic version: {pydantic.__version__}")


LangChain version: 1.4.3
LangChain Core version: 1.6.6
Pydantic version: 2.13.5


### 2. استعراض طبقة الـ Middleware الجديدة

من أقوى إضافات LangChain v1+ هي الـ **Middlewares**، ودي طبقات جاهزة بتتحكم في سلوك الوكيل (زي إيقاف الوكيل لأخذ موافقة البشري، أو تقليل طول السياق). تعال نشوف المتاح منها في المكتبة!


In [3]:
import langchain.agents.middleware as mw

# عرض أبرز الـ Middlewares الجاهزة للاستخدام
built_in_mw = [name for name in dir(mw) if "Middleware" in name]
print(f"عدد الـ Middlewares الجاهزة: {len(built_in_mw)}")
for item in built_in_mw:
    print(" -", item)


عدد الـ Middlewares الجاهزة: 16
 - AgentMiddleware
 - ContextEditingMiddleware
 - FilesystemFileSearchMiddleware
 - HumanInTheLoopMiddleware
 - LLMToolSelectorMiddleware
 - ModelCallLimitMiddleware
 - ModelFallbackMiddleware
 - ModelRetryMiddleware
 - PIIMiddleware
 - ProviderToolSearchMiddleware
 - ShellToolMiddleware
 - SummarizationMiddleware
 - TodoListMiddleware
 - ToolCallLimitMiddleware
 - ToolErrorMiddleware
 - ToolRetryMiddleware


### 3. تجربة أنواع الرسائل الأربعة (Messages Hierarchy)

أي وكيل ذكاء اصطناعي بيتكلم معاك ومع الأدوات الخارجية من خلال 4 أنواع قياسية من الرسائل. تعال نبني محادثة بسيطة فيها سؤال واستدعاء أداة ورد!


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages_history = [
    SystemMessage(content="أنت مساعد برمجي ذكي تشرح المفاهيم ببساطة."),
    HumanMessage(content="ما هي حالة الطقس في القاهرة الآن؟"),
    AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "Cairo"}, "id": "call_1"}]),
    ToolMessage(content="درجة الحرارة 26 مئوية والجو مشمس.", tool_call_id="call_1"),
    AIMessage(content="الطقس في القاهرة الآن مشمس ودرجة الحرارة 26 مئوية.")
]

for m in messages_history:
    print(f"[{type(m).__name__}]: {m.content or 'طلب استدعاء أداة...'}")


[SystemMessage]: أنت مساعد برمجي ذكي تشرح المفاهيم ببساطة.
[HumanMessage]: ما هي حالة الطقس في القاهرة الآن؟
[AIMessage]: طلب استدعاء أداة...
[ToolMessage]: درجة الحرارة 26 مئوية والجو مشمس.
[AIMessage]: الطقس في القاهرة الآن مشمس ودرجة الحرارة 26 مئوية.


### 4. جرب بنفسك!

دلوقتي دورك في التجربة الحرة:
1. جرب تغير الرسالة البشرية في الكود تحت وتضيف رسالة `HumanMessage` جديدة وتطبع محتواها ونوعها.
2. جرب تستكشف دوال `init_chat_model` أو `create_agent` وتطبع التوثيق السريع الخاص بها (`__doc__`) لتتعرف على معاملاتها!


In [5]:
# جرب بنفسك هنا
from langchain.chat_models import init_chat_model

my_question = HumanMessage(content="هل أقدر أشغل Groq و OpenAI بنفس الدالة في LangChain v1؟")
print(f"رسالتك ({type(my_question).__name__}): {my_question.content}")

# استعراض أول 5 سطور من التوثيق المدمج لدالة تهيئة النماذج الموحدة
print("\nنبذة عن دالة init_chat_model:")
print(init_chat_model.__doc__[:300] + "...")


رسالتك (HumanMessage): هل أقدر أشغل Groq و OpenAI بنفس الدالة في LangChain v1؟

نبذة عن دالة init_chat_model:
Initialize a chat model from any supported provider using a unified interface.

    **Two main use cases:**

    1. **Fixed model** – specify the model upfront and get a
        ready-to-use chat model.
    2. **Configurable model** – choose to specify parameters
        (including model name) at ru...
